# The risky route to class: traffic crashes near schools in New York

How dangerous is it to walk to school? In this workshop we use two public datasets from New York City, police reports of traffic crashes and the list of schools, to find out how many crashes involving pedestrians happen close to schools, which schools are the most exposed, and at what time of day.

You don't need to know how to code. For each step there is a prompt: you copy it into ChatGPT or Claude, paste the code you get into the empty cell below the prompt, and run it. If it doesn't work, you send the error back to the chat. Most of the work is in asking clearly and checking whether the result makes sense.

This notebook is based on [The Risky Route to Class](https://teodoracurcic.github.io/lede2025-risky-route-to-class/index.html), a project that did the same analysis for Belgrade. It combined nine years of police accident data with school addresses from the Ministry of Education, and found that two out of three recorded traffic accidents in Belgrade happened within 250 meters of a school, most of them in the city center where schools are closest together. At the end you can compare the two cities.

**Data**

- [Motor Vehicle Collisions - Crashes](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95), NYPD, published on NYC Open Data
- [2019 - 2020 School Locations](https://data.cityofnewyork.us/Education/2019-2020-School-Locations/wg9x-4ke6), NYC Department of Education, published on NYC Open Data

**Contents**

0. Python basics
1. How to work with the AI
2. Load the crash data
3. Clean the crash data
4. Load and clean the school data
5. Crashes within 250 meters of a school
6. Which schools are the most exposed
7. Export for maps

## Before you start: install the libraries

Run the cell below once. It installs the tools we use today. If they're already installed, it just says so and nothing changes.

The *kernel* is the Python that runs in the background of the notebook: it runs your cells and remembers everything you've created so far, like tables and variables. Restarting it clears that memory, so after a restart you run the cells again from the top.

After it finishes, **restart the kernel** so the notebook picks up the new tools. In the browser version of Jupyter that's the menu *Kernel > Restart*; in VS Code it's the *Restart* button at the top of the notebook. Then continue from the next cell.

We use `%pip` and not `!pip` on purpose: `%pip` installs into the same Python the notebook is running on. With `!pip` the tools sometimes end up in a different Python on your computer and the notebook still can't find them.

In [ ]:
%pip install pandas "geopandas>=1.0"

## 0. Python basics

If you've used Python before, skip to part 1.

### Where you can run Python

There are a few ways to work with a notebook like this one, and they all do the same job:

- **Jupyter Notebook or JupyterLab** in your browser. You start it by typing `jupyter notebook` (or `jupyter lab`) in the terminal, and it opens a tab in your browser. If you installed Python through Anaconda, it's already there.
- **VS Code**, a free code editor from Microsoft. Install the *Python* and *Jupyter* extensions, then just open the `.ipynb` file. You don't need to start Jupyter from the terminal first, VS Code does that for you. The first time, it will ask you to pick a *kernel* in the top right corner: choose the Python version where you installed the libraries.
- **Google Colab** ([colab.research.google.com](https://colab.research.google.com)). It runs in the browser on Google's computers, so there's nothing to install and you only need a Google account. Upload this notebook with *File > Upload notebook*. Keep in mind that files you download there disappear when you close the session.
- Plain Python scripts (`.py` files) run from the terminal. That's how most code runs in production, but for exploring data, notebooks are easier because you see the result after every step.

### How a notebook works

A notebook is made of *cells*. Some are text, like this one, and some are code. To run a code cell, click on it and press **Shift + Enter**. The result shows up under the cell.

Cells run one at a time, and order matters: if a cell uses something from an earlier cell, the earlier one has to be run first. If you get lost, restart the kernel and run everything again from the top.

Try it on the cell below.

In [ ]:
print("Hello, newsroom!")

A **variable** is a name you give to a value so you can use it later. Text goes in quotes, numbers don't.

In [ ]:
city = "New York"
radius_meters = 250

print(city)
print(radius_meters * 2)

A **list** holds several values in order, in square brackets. A **dictionary** pairs a key with a value, in curly brackets. We'll use dictionaries to translate codes into names.

In [ ]:
years = [2023, 2024, 2025]
print(years)

borough_names = {"K": "Brooklyn", "M": "Manhattan"}
print(borough_names["K"])

**Libraries** are toolboxes other people wrote, and `import` loads them. The one we use most is **pandas**, which works with tables. In pandas a table is called a **DataFrame**.

In [ ]:
import pandas as pd

example = pd.DataFrame({
    "school": ["PS 1", "PS 2", "PS 3"],
    "crashes_nearby": [12, 3, 27],
})
example

You do things to a table by writing them after a dot. `.shape` tells you how many rows and columns there are, and `.sort_values()` sorts the table.

In [ ]:
print(example.shape)
example.sort_values("crashes_nearby", ascending=False)

**Errors are normal**, even for people who code every day. When something goes wrong, Python prints a message, and the useful part is usually the **last line**. The cell below has a mistake on purpose. Run it and read the last line, then compare it with the column names in the table above.

In [ ]:
example["School"]

`KeyError: 'School'` means there's no column called `School`. Python cares about capital letters, and our column is `school`. When an error is harder to read than this one, copy the whole thing into the chat and ask what it means.

## 1. How to work with the AI

**How each step works.** Every step below has a prompt and an empty code cell under it. Copy the prompt into ChatGPT or Claude, paste the code you get back into the empty cell, and run it. If you get an error, copy the whole error into the chat together with a sentence like "I got this error when I ran your code". Keep the whole workshop in **one chat**, so the AI remembers the table and column names from earlier steps.

**Four rules worth remembering:**

1. **Describe the data, don't upload it.** The AI only needs to know what the table looks like: the column names, a few example rows and the column types. You also shouldn't paste sensitive data into a chatbot.
2. **Say what you want to find out, not which function to use.** "Count crashes per hour" works better than guessing what the code should look like.
3. **Be specific about names.** If you tell the AI what to call the new table or column, the next steps will fit together.
4. **Check every result.** Count the rows before and after each step, and look at a few examples yourself. The AI knows what your columns are called, but not what they mean.

Start the chat with the prompt below. It sets the rules for the whole conversation, so you don't have to repeat them in every step.

**Prompt**

> I'm a journalist and I'm new to Python. I'm working in a Jupyter Notebook and I want to analyze
> traffic crashes near schools in Brooklyn, New York, using pandas and geopandas.
> I'll ask you for code one step at a time. For every step:
> - give me the code in one block that I can paste into a single notebook cell
> - add short comments in plain English that explain what the code does, but only on lines that
>   actually do something (no comments on closing brackets) and keep the code as short as you can
> - import a library only the first time we need it; don't repeat imports from earlier steps,
>   because I run all the cells in order
> - use exactly the table and column names I give you
> - only use pandas, geopandas and Python's standard library, unless I ask for something else
> - after the code, tell me in one or two sentences what I should see when I run it
> Reply "ready" if this is clear.

## 2. Load the crash data

The full NYC crash dataset has more than two million rows, which is too much for a laptop. We don't need all of it, so we ask the NYC Open Data website for only the part we want: crashes in Brooklyn, one file per year. The files are saved in a folder called `data`, so if you run the cell again, nothing gets downloaded twice.

**Prompt**

> Step 1: download the data.
> I want to download traffic crash data from NYC Open Data. The dataset is called
> "Motor Vehicle Collisions - Crashes" and its API address is
> https://data.cityofnewyork.us/resource/h9gi-nx95.csv
>
> The whole dataset is too big, so I only want crashes in Brooklyn, saved as one file per year
> for 2023, 2024 and 2025.
> - The borough is in the column `borough`, written in capital letters, for example 'BROOKLYN'.
> - The date is in the column `crash_date`.
> - Use the API parameter $where to filter by borough and by date.
> - Set $limit to 200000, because without it the API only returns 1,000 rows.
> - The filter has spaces and quotes in it, so make sure the web address is encoded correctly.
>
> Save each year as data/crashes_brooklyn_YEAR.csv. Create the data folder if it doesn't exist,
> and skip the download if the file for that year is already there.
> Print how many rows were downloaded for each year.

Now we put the three yearly files together. You can use the same approach for any folder full of files with the same columns, for example monthly reports.

**Prompt**

> Step 2: combine the files.
> In the folder `data` I now have three files: crashes_brooklyn_2023.csv, crashes_brooklyn_2024.csv
> and crashes_brooklyn_2025.csv. They all have the same columns.
> Find every file in that folder whose name starts with crashes_brooklyn_ and ends with .csv,
> read them all and stack them into one DataFrame called `crashes`.
> Print the list of files you found and the number of rows and columns in `crashes`.

Before we change anything, let's look at the table. The output of the next cell is exactly what the AI needs if you get stuck later: copy it into the chat whenever you need help with this table.

**Prompt**

> Step 3: look at the data.
> Show me the first 3 rows of `crashes`, and then the data type of every column.

## 3. Clean the crash data

### Keep only what we need

The table has 29 columns. Most of them we won't use, and a smaller table is easier to read and to describe to the AI.

**Prompt**

> Step 4: keep only the columns we need.
> My DataFrame `crashes` has 29 columns. Keep only these 12, in this order:
> collision_id, crash_date, crash_time, latitude, longitude, on_street_name, cross_street_name,
> number_of_persons_injured, number_of_persons_killed, number_of_pedestrians_injured,
> number_of_pedestrians_killed, contributing_factor_vehicle_1.
> Keep the result in the same variable `crashes` and show the first 2 rows.

### Dates and times

Right now the date and time are just text. Once Python understands them as a date and a time, we can count crashes by year, by day of the week and by hour.

**Prompt**

> Step 5: dates and times.
> In `crashes`, the column crash_date has text like "2024-03-15T00:00:00.000" and crash_time has text
> like "9:05" or "14:30". Hours before 10 have no leading zero, and a few times might be missing.
> - Convert crash_date into a real date.
> - Add a column `year` with the year.
> - Add a column `weekday` with the full English name of the day (Monday, Tuesday...).
> - Add a column `hour` with the hour as a number from 0 to 23. If the time is missing, leave it empty
>   instead of stopping with an error.
> Then show me the first 5 rows of crash_date, crash_time, year, weekday and hour.

**Check:** each year should have roughly the same number of crashes. If one year has far fewer, the download was probably cut off. It doesn't mean the streets got safer.

**Prompt**

> Step 6: check the years.
> Count how many crashes there are in each year in `crashes`, sorted by year.

### Missing and wrong locations

We can only put a crash on a map if we know where it happened. Some crashes in this dataset have no coordinates, and some have 0, which on a map is a spot in the ocean off the coast of West Africa.

**Prompt**

> Step 7: clean the coordinates.
> In `crashes`, the columns latitude and longitude are numbers. Some rows have them empty and some
> have them set to 0.
> - Print the total number of rows.
> - Print how many rows have an empty latitude and how many have latitude equal to 0.
> - Then keep only rows inside New York City: latitude between 40.4 and 41 and longitude between
>   -74.3 and -73.6. This also removes the empty and zero values.
> - Keep the result in `crashes` and print how many rows are left.

Write down how many crashes you lost here. It belongs in the methodology of your story, something like: "About 2% of crashes could not be mapped because the police didn't record where they happened."

### Crashes with pedestrians

The story is about walking to school, so from here on we only look at crashes where a pedestrian was injured or killed.

**Prompt**

> Step 8: find crashes with pedestrians.
> In `crashes`, add a column `pedestrian` that is True when number_of_pedestrians_injured is more
> than 0 or number_of_pedestrians_killed is more than 0, and False otherwise.
> Then show how many crashes are True and False, first as counts and then as percentages.

**Prompt**

> Step 9: a first look at pedestrian crashes.
> Make a new DataFrame `pedestrian_crashes` with only the rows of `crashes` where `pedestrian` is True.
> Then show:
> 1. the 10 most common values in the column contributing_factor_vehicle_1
> 2. the number of pedestrian crashes for every hour from 0 to 23, sorted by hour

One of the most common "causes" is `Unspecified`. That's not a cause, it means the police officer didn't fill in that field. Before you write "the most common cause of crashes is...", check how big that group is.

## 4. Load and clean the school data

The school list is small, about 2,000 rows, so we download all of it at once. It's from the 2019/2020 school year. That's fine for practice, but for a published story you'd look for a newer list or ask the Department of Education.

**Prompt**

> Step 10: download the schools.
> Download the list of New York City schools from https://data.cityofnewyork.us/resource/wg9x-4ke6.csv
> Add $limit=5000 to the address so I get all the rows.
> Save it as data/schools.csv, and skip the download if that file already exists.
> Then read the file into a DataFrame called `schools`, print the number of rows and columns,
> and show the first 2 rows.

### Drop what we don't need

The file also has the names and phone numbers of school principals. We don't need them, and it's a good habit not to keep personal data you don't use.

**Prompt**

> Step 11: keep and rename columns.
> From `schools`, keep only these 7 columns and rename them like this:
> system_code -> code
> location_name -> name
> location_category_description -> category
> primary_address_line_1 -> address
> nta_name -> neighborhood
> latitude -> latitude
> longitude -> longitude
> Keep the result in `schools` and show the first 3 rows.

### The borough is hidden in the school code

There's no borough column, but every school code has the borough in it. In a code like `13K001`, the letter tells you where the school is: M is Manhattan, X the Bronx, K Brooklyn, Q Queens and R Staten Island.

This happens a lot with official data: what you need is there, just not where you'd expect it.

**Prompt**

> Step 12: find the borough.
> In `schools`, the column `code` has values like "13K001" or "02M047". The letter in the middle
> is the borough: M = Manhattan, X = Bronx, K = Brooklyn, Q = Queens, R = Staten Island.
> Add a column `borough` with the full borough name.
> Then show how many schools each borough has, and also show how many rows have no borough,
> in case some codes look different.

**Prompt**

> Step 13: keep Brooklyn schools that have a location.
> In `schools`, make sure latitude and longitude are numbers (anything that isn't a valid number
> should become empty). Then make a new DataFrame `brooklyn_schools` with only the schools where
> borough is "Brooklyn" and both latitude and longitude are filled in.
> Print how many schools are left, and show how many schools there are of each category
> (column `category`).

## 5. Crashes within 250 meters of a school

So far we've worked with ordinary tables. To measure distances we turn them into **GeoDataFrames**, which are tables where every row also knows its place on the map.

A word about coordinates. Latitude and longitude are measured in degrees (the system is called EPSG:4326, it's what GPS and web maps use). You can't draw a circle of 250 meters in degrees, so we switch to a system that measures in meters. For New York that's UTM zone 18N, code EPSG:32618. The Belgrade project used UTM zone 34N (EPSG:32634) for the same reason. For your own city, ask the AI which zone to use.

**Prompt**

> Step 14: put the data on the map.
> I have two DataFrames with latitude and longitude columns: `pedestrian_crashes` and `brooklyn_schools`.
> Turn each one into a GeoDataFrame with point geometry built from those columns. The coordinates are
> in EPSG:4326. Remember that x is longitude and y is latitude.
> Then convert both to EPSG:32618, so distances are measured in meters.
> Call them `crashes_gdf` and `schools_gdf`, and print the coordinate system and number of rows of each.

Mixing up longitude and latitude is one of the most common mapping mistakes, and the AI makes it too. If your points end up in Antarctica or in the middle of the ocean, check this first.

### A circle around every school

**Prompt**

> Step 15: draw circles around the schools.
> From `schools_gdf`, make a new GeoDataFrame `school_zones` where every school is a circle with a
> radius of 250 meters instead of a point. Keep the columns code, name, neighborhood and category.
> Store the radius in a variable called RADIUS, so I can change it later. Show the first 2 rows.

### Which crashes fall inside a circle?

A **spatial join** connects two tables by location rather than by a shared column. For every crash, it finds every school circle that the crash is inside. One crash can be close to several schools, so we have to be careful not to count it more than once.

**Prompt**

> Step 16: find crashes near schools.
> I have `crashes_gdf` (pedestrian crashes as points) and `school_zones` (250 m circles around schools),
> both in EPSG:32618. Do a spatial join that finds every crash inside at least one circle, and call
> the result `crashes_near_schools`.
> One crash can be near several schools, so make a list of unique collision_id values from the result
> and call it `near_ids`.
> Then print one sentence like: "X of Y pedestrian crashes (Z%) happened within 250 m of a school",
> where Y is the number of rows in `crashes_gdf`.

**Before you publish this number, ask yourself whether it's actually surprising.** In a dense city, a big part of the map is close to some school. If school zones cover 40% of the borough and 40% of crashes happen inside them, schools aren't more dangerous than anywhere else. The number only means something when you compare it with the share of the area.

For a rough outline of the borough we use the area that all the crashes cover. It's a quick check for the workshop, not a number to publish.

**Prompt**

> Step 17: compare with the area.
> Merge all the circles in `school_zones` into one shape (so overlapping circles are counted once)
> and calculate its area. Then calculate the area of the convex hull around all the points in
> `crashes_gdf`, as a rough outline of the borough.
> Print what percentage of that area is covered by school zones, and next to it the percentage of
> pedestrian crashes that happened in school zones (the number of `near_ids` divided by the number
> of rows in `crashes_gdf`).

### During school hours

A crash at three in the morning matters less for this story than one at eight on a Tuesday. Let's look at weekday mornings and afternoons, when children are on their way to and from school.

**Prompt**

> Step 18: school hours.
> In `crashes_gdf` I have the columns `weekday` (Monday, Tuesday...) and `hour` (0 to 23).
> - Add a column `school_hours` that is True when the crash happened Monday to Friday and the hour
>   is 7 or 8 (morning) or 14 or 15 (afternoon).
> - Add a column `near_school` that is True when the crash's collision_id is in my list `near_ids`.
> - Then make a table that crosses near_school with school_hours, with totals for rows and columns.

## 6. Which schools are the most exposed?

**Prompt**

> Step 19: count crashes per school.
> `crashes_near_schools` has one row for every crash-school pair, with the school columns code, name,
> neighborhood and category, and the crash columns collision_id, number_of_pedestrians_injured and
> number_of_pedestrians_killed.
> For each school, calculate:
> - pedestrian_crashes: the number of unique collision_id values
> - pedestrians_injured: the sum of number_of_pedestrians_injured
> - pedestrians_killed: the sum of number_of_pedestrians_killed
> Call the result `per_school`, sort it from the most crashes to the fewest, and show the top 15.

**Check:** schools with no crashes nearby aren't in this table at all, because the spatial join only keeps matches. Before you write about "the safest school", you have to add them back with a count of 0.

**Prompt**

> Step 20: add the schools with no crashes.
> Merge `schools_gdf` (all Brooklyn schools) with the columns code, pedestrian_crashes,
> pedestrians_injured and pedestrians_killed from `per_school`, using the column `code`, and keep
> every school from `schools_gdf`. Schools that aren't in `per_school` should get 0 in those three
> columns, as whole numbers. Call the result `schools_with_counts`.
> Print how many schools have 0 pedestrian crashes within 250 m.

Several schools often share one building. If the top of your list has three schools at the same address, that's one dangerous corner, not three.

**Prompt**

> Step 21: schools in the same building.
> In `schools_with_counts`, count how many schools share each address (column `address`) and show
> the 10 addresses with the most schools.

**Prompt**

> Step 22: neighborhoods.
> Using `crashes_near_schools`, count how many unique crashes (collision_id) happened near schools in
> each neighborhood (column `neighborhood`). A crash near two schools in the same neighborhood should
> count once. Show the 10 neighborhoods with the most crashes.

## 7. Export for maps

GeoJSON is a file format that Datawrapper, Flourish, QGIS and Mapbox can all open. Web maps expect coordinates in degrees, so we switch back to EPSG:4326 before saving.

**Prompt**

> Step 23: save the files.
> Save two GeoJSON files, both converted back to EPSG:4326:
> 1. `schools_with_counts` as brooklyn_schools_crashes.geojson
> 2. only the rows of `crashes_gdf` where near_school is True, as brooklyn_crashes_near_schools.geojson.
>    The column crash_date is a date, and GeoJSON can't store dates, so turn it into text
>    (year-month-day) first.
> Print how many schools and how many crashes were saved.

## Before you publish

- Write down how many rows you removed at each step and why (missing locations, wrong values).
- Make sure you know what every column you use actually means. If the documentation doesn't say, ask whoever published the data.
- Compare your main number with something, like the share of the area in step 17, so you know whether it's surprising.
- Look at the top results yourself, on a map, and if you can, on the street.
- If there's a line of code you don't understand, ask about it:

**Prompt**

> Explain this code to me line by line as if I've never programmed before.
> Then tell me what could go wrong with it, and how I could check that the result is correct.
> [paste the code]

## Where to go from here

- Run the same analysis for another borough: change 'BROOKLYN' in step 1 and "Brooklyn" in step 13.
- Compare elementary schools and high schools. Are younger children more exposed?
- Look at how the numbers change from year to year.
- Try it on your own city. You need two things: crashes with locations, and schools with locations or addresses. If you only have addresses, ask the AI how to turn them into coordinates with the geopy library.